# R07 — Rumble Comparison

**Goal:** Compare our local performance against the same opponents' performance
in the rumble dataset.

- Our hit rate vs the opponent's typical hit rate in the rumble
- Opponent's hit rate against us vs their average hit rate in the rumble
- Do opponents play differently against us than the average competitor?

In [1]:
import sys; sys.path.insert(0, '..')
from retrospective._retro_helpers import load_local_scores, add_opponent_names
from _loader import build_robot_index, load_stratified, CSV_ROOT_DEFAULT
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

# Load local scores
local_scores = add_opponent_names(load_local_scores())
print(f'Local scores: {len(local_scores):,} rows')

# Load rumble scores for comparison
try:
    rumble_selection = build_robot_index(csv_root=CSV_ROOT_DEFAULT, max_robots=50, battles_per_robot=10)
    rumble_scores = load_stratified('scores.csv', rumble_selection, csv_root=CSV_ROOT_DEFAULT)
    print(f'Rumble scores: {len(rumble_scores):,} rows')
except Exception as e:
    print(f'Could not load rumble data: {e}')
    rumble_scores = pd.DataFrame()

Indexed 60 ticks.csv files across 7 distinct robots from 1 root(s).
Selected 7 robots × ~100 battles = 60 (battle, robot) pairs to load.
Loaded 60 scores.csv files → 2,100 rows × 17 cols, 7 robots (~0.4 MB)
Local scores: 1,050 rows
⚠ CSV root does not exist: ..\output\csv
Indexed 0 ticks.csv files across 0 distinct robots from 1 root(s).
Selected 0 robots × ~10 battles = 0 (battle, robot) pairs to load.
⚠ No scores.csv files found for the given selection.
Rumble scores: 0 rows


In [2]:
# Compare: our local stats per opponent vs rumble averages
if len(local_scores) > 0 and len(rumble_scores) > 0:
    # Local: per-opponent stats
    local_stats = local_scores.groupby('opponent_name').agg(
        local_our_hr=('our_hit_rate', 'mean'),
        local_opp_hr=('opponent_hit_rate', 'mean'),
        local_net_dmg=('net_damage', 'mean'),
        local_rounds=('round', 'count'),
    ).reset_index()
    
    # Rumble: per-robot averages (each robot as observer)
    rumble_stats = rumble_scores.groupby('robot_name').agg(
        rumble_our_hr=('our_hit_rate', 'mean'),
        rumble_opp_hr=('opponent_hit_rate', 'mean'),
        rumble_net_dmg=('net_damage', 'mean'),
        rumble_rounds=('round', 'count'),
    ).reset_index()
    
    # Match opponents (local opponent_name is e.g. 'Shadow 3.83c', rumble robot_name similar)
    # Try fuzzy match on the bot family name (first word before space)
    local_stats['match_key'] = local_stats['opponent_name'].str.split().str[0]
    rumble_stats['match_key'] = rumble_stats['robot_name'].str.split().str[0]
    
    merged = local_stats.merge(rumble_stats, on='match_key', how='inner')
    print(f'Matched {len(merged)} opponents between local and rumble data.')
    
    if len(merged) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Our hit rate vs rumble average
        ax = axes[0]
        ax.scatter(merged['rumble_our_hr']*100, merged['local_our_hr']*100, alpha=0.7, s=80, color='steelblue')
        for _, row in merged.iterrows():
            ax.annotate(row['match_key'], (row['rumble_our_hr']*100, row['local_our_hr']*100), fontsize=7, alpha=0.7)
        lim = max(merged['rumble_our_hr'].max(), merged['local_our_hr'].max())*100 + 5
        ax.plot([0, lim], [0, lim], '--', color='red', alpha=0.5, label='Equal')
        ax.set_xlabel('Opponent avg hit rate in rumble (%)')
        ax.set_ylabel('Our hit rate against opponent (%)')
        ax.set_title('Our aim vs how accurate opponents typically are')
        ax.legend()
        
        # Opponent hit rate vs us compared to rumble avg
        ax = axes[1]
        ax.scatter(merged['rumble_opp_hr']*100, merged['local_opp_hr']*100, alpha=0.7, s=80, color='coral')
        for _, row in merged.iterrows():
            ax.annotate(row['match_key'], (row['rumble_opp_hr']*100, row['local_opp_hr']*100), fontsize=7, alpha=0.7)
        lim = max(merged['rumble_opp_hr'].max(), merged['local_opp_hr'].max())*100 + 5
        ax.plot([0, lim], [0, lim], '--', color='red', alpha=0.5, label='Equal')
        ax.set_xlabel('Opponents get hit rate in rumble (%)')
        ax.set_ylabel('Opponent hit rate vs us (%)')
        ax.set_title('How hard opponents hit us vs how hard they get hit in rumble')
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        print('\nComparison table:')
        display_cols = ['match_key', 'local_our_hr', 'rumble_our_hr', 'local_opp_hr', 'rumble_opp_hr', 'local_net_dmg', 'rumble_net_dmg']
        print(merged[display_cols].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    else:
        print('No matching opponents found between local and rumble datasets.')
else:
    print('Need both local and rumble data for comparison.')

Need both local and rumble data for comparison.
